In [ ]:
!pip install matplotlib-venn
!pip install datasets transformers evaluate
!pip install sentence-transformers
!pip install fsspec==2023.6.0
!pip install rouge_score



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [ ]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets evaluate tqdm pandas numpy


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121


In [ ]:
import torch
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from datasets import load_dataset
import evaluate
from tqdm import tqdm
import numpy as np
import pandas as pd


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def prepare_cnndm_dataset():
    dataset = load_dataset("cnn_dailymail", "3.0.0")

    def preprocess_function(examples):
        inputs = ["summarize: " + doc for doc in examples["article"]]
        targets = examples["highlights"]
        return {"inputs": inputs, "targets": targets}

    dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["article", "highlights", "id"]
    )
    return dataset["train"], dataset["validation"], dataset["test"]

train_dataset, val_dataset, test_dataset = prepare_cnndm_dataset()


tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn").to(device)


def data_collator(batch):
    inputs = [item["inputs"] for item in batch]
    targets = [item["targets"] for item in batch]

    model_inputs = tokenizer(
        inputs,
        max_length=1024,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=256,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        ).input_ids

    labels[labels == tokenizer.pad_token_id] = -100

    return {
        "input_ids": model_inputs["input_ids"],
        "attention_mask": model_inputs["attention_mask"],
        "labels": labels
    }

def train_model(model, train_dataset, val_dataset, epochs=1, batch_size=4):
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator,
        pin_memory=True
    )

    optimizer = AdamW(model.parameters(), lr=5e-5)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=100,
        num_training_steps=len(train_loader) * epochs
    )

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc="Training"):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

            if len(train_loader) % 10 == 0:
                print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**2:.2f}MB")

        avg_loss = total_loss / len(train_loader)
        print(f"Average Loss: {avg_loss:.4f}")

    return model

print("\nFine-tuning BART on CNN/DailyMail...")
model = train_model(
    model,
    train_dataset.select(range(1000)),
    val_dataset.select(range(200)),
    epochs=3,
    batch_size=4
)

rouge = evaluate.load("rouge")

def evaluate_model(model, test_dataset, num_samples=100):
    test_subset = test_dataset.select(range(num_samples))
    predictions, references = [], []

    model.eval()
    with torch.no_grad():
        for example in tqdm(test_subset, desc="Evaluating"):
            inputs = tokenizer(
                example["inputs"],
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            ).to(device)

            summary_ids = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=256,
                num_beams=4,
                early_stopping=True
            )

            predictions.append(tokenizer.decode(summary_ids[0], skip_special_tokens=True))
            references.append(example["targets"])

    return rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )


print("\nEvaluating BART...")
results = evaluate_model(model, test_dataset, num_samples=50)


results_df = pd.DataFrame({
    "Model": ["BART-large-cnn"],
    "ROUGE-1": [results["rouge1"]],
    "ROUGE-2": [results["rouge2"]],
    "ROUGE-L": [results["rougeL"]]
})

print("\nEvaluation Results:")
print(results_df)

Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

...

Training: 100%|██████████| 250/250 [09:07<00:00,  2.19s/it]

GPU Memory: 6428.36MB
Average Loss: 0.3702



Evaluating BART...


Evaluating: 100%|██████████| 50/50 [01:23<00:00,  1.67s/it]



Evaluation Results:
            Model   ROUGE-1   ROUGE-2   ROUGE-L
0  BART-large-cnn  0.357689  0.147245  0.264963
